# Script pipeline

*Circularity paradox* workflow:

1. `RAW/scraped_data.xlsx` → Ingest & clean raw marketplace data → `PROC/df_cleaned_en.parquet`, `DER/df_cleaned_en_described.csv`   
2. `PROC/df_cleaned_en.parquet` → Category & market summaries → `PROC/product_categories_overview.parquet`, `DER/*`
3. `PROC/df_cleaned_en.parquet`, `processed/product_categories_overview.parquet`, `FAC/retail_prices_representative_products.yaml` → Representative products, prices, and economic benefits 
→ `PROC/earnings_savings.parquet`, `DER/*`

df = pd.read_parquet(PROC/'df_cleaned_en.parquet')
cats = pd.read_parquet(PROC/'product_categories_overview.parquet')

rp, earnings_savings, totals, econ_ben_stats = make_representatives_and_prices(
    df, cats,
    retail_config_path=str(FAC/'retail_prices.yaml'),   
    representative_selector="mean",                     # or "median"
    low_retained_thresh=5.0,
    high_retained_thresh=120.0
)

write_df(rp,     DER/'representative_products.parquet')
write_df(earnings_savings, PROC/'earnings_savings.parquet')
write_df(totals, DER/'total_values_per_category.parquet')
write_df(econ_ben_stats, DER/'ecom_ben.parquet')


4. Household CF factors for Sweden → `factors/*`  
5. Potential environmental benefits → `derived/SI_V_pot_env_ben.parquet`
6. Economic and environmental benefits (results)  
7. Re‑spending effects → `derived/df_re_spending_effect_{agg,disagg}.parquet`  
8. Circular rebound effect computation → `results/df_cre.parquet`

**Folders created automatically:** `raw/ processed/ derived/ factors/ results/ exports/ diagnostics/`  
**Convention:** internal Parquet; CSV/Excel only in `exports/`

**Initial files needed:** 
`raw/scraped_data.xlsx` # Raw data scraped from the marketplace.
    
`factors/df_32cat_CFexio_RL.xlsx` # 32 marketplace product categories matched with the 200 EXIOBASE product categories + impact factor (GHG emissions per M EUR).
In the cases where the matches were one-to-many, an average value of the environmental stressors is calculated.    

`factors/Exio_COICOP12.xlsx` # Concordance file matching the 200 EXIOBASE product categories with 12 COICOP categories.

`factors/EXIOBASE341f_CC500f.csv` # Characterization file used to convert envitonmental stressors into impact factors [GHG emissions (GWP100)|kg CO2 eq.|Problem oriented approach: baseline (CML, 2001)|GWP100 (IPCC, 2007)]

`factors/retail_prices_representative_products.yaml` # Retail price of the 32 representative products.

## 00 Scraped raw data

Do not run it

In [ ]:
#website has changed its html structure, therefore this script does not longer work 
import requests
import bs4
from bs4 import BeautifulSoup
import xlsxwriter
import re
import numpy as np
import six

workbook = xlsxwriter.Workbook('dataset/data_privat_collected.xlsx')
worksheet = workbook.add_worksheet()
bold = workbook.add_format({'bold': 1})
worksheet.set_column(0, 0, 50)
worksheet.set_column(1, 1, 30)
worksheet.set_column(2, 2, 20)
worksheet.set_column(3, 3, 20)
worksheet.write_string(0, 0, "Item", bold)
worksheet.write_string(0, 1, "Price", bold)
worksheet.write_string(0, 2, "Category", bold)
worksheet.write_string(0, 3, "Region", bold)

row = 1
col = 0

url = 'https://www.marketplace.se/hela_sverige?ca=11&st=s&f=p&w=3' #url modified to marketplace.se to preserve anonymity

while not url.endswith('&last=1'):  
    
    response = requests.get(url)    
    try:    
        response.raise_for_status() 
    except Exception as exc:    
        print('There was a problem: %s' % (exc))

    soup = BeautifulSoup(response.text, "html.parser")

    items = soup.find_all("a", "item_link")
    prices = soup.find_all("p", "list_price") 
    categories = soup.find_all('a', {'tabindex': '-1'})
    regions = soup.find_all('div', 'pull-left')

    for item_a, price_p, category_a, region_div in zip(items, prices, categories, regions[6:]):
         item = item_a.string.strip()   

         price = price_p.text
         if not price:
             price = "NULL"

         category = category_a.text
         if not(re.search(r"Lägenheter|Utland|Djur|Villor|Tjänster|Lokaler & fastigheter|Affärsöverlåtelser|Upplevelser & nöje|Tomter|Gårdar|Efterlysningar|Fritidsboende|Radhus", category)) and not(re.search(r"Jobb", region_div.text)):
             region = region_div.text.split(',')[-1]          
             worksheet.write_string(row, col, item)
             worksheet.write_string(row, col + 1, price)
             worksheet.write_string(row, col + 2, category)        
             worksheet.write_string(row, col + 3, region)             
             row += 1     
         
    nextLink = soup.find_all('a', 'page_nav')[5]
    if not "Nästa sida »" in nextLink.decode_contents().strip():
        nextLink = soup.find_all('a', 'page_nav')[6]
        if not "Nästa sida »" in nextLink.decode_contents().strip():  
            nextLink = soup.find_all('a', 'page_nav')[7]               
    
    nextLinkSuffix = nextLink.get('href')     
    url = 'https://www.marketplace.se/hela_sverige' + nextLinkSuffix     #url modified to marketplace.se to preserve anonymity

workbook.close() 

There was a problem: 404 Client Error: Not Found for url: https://marketplace.se/hela_sverige?ca=11&st=s&f=p&w=3


IndexError: list index out of range

In [ ]:
# This is the supporting material I
df_raw = pd.read_excel(RAW / 'scraped_data.xlsx')
df_raw